In [ ]:
import pandas as pd
import os

# import datasets
debt_to_gdp = pd.read_csv('GDD_annual.csv')
us_rates = pd.read_csv('GS10_monthly.csv')
trade = pd.read_csv('monthly_trade.csv')
sov_default = pd.read_excel('DEFAULT_DATABASE.xlsx', sheet_name = 'DATASET Defaults & Restruct.', skiprows=5)
exchange_rates = pd.read_csv('exchange_rates.csv')
cpi_data = pd.read_csv('CPI_data.csv')

# Modify the imported datasets to prepare for merging
# To-do notes:
-> Find a fitting treatment variable
-> think of design of default variable#
-> start



# Modify GDD dataset


In [ ]:
# debt to gdp
display(debt_to_gdp.head())

,COUNTRY,INDICATOR,FREQUENCY,TIME_PERIOD,OBS_VALUE,SCALE
0,Cambodia,"Debt instruments, Central government, Percent ...",Annual,1975,NaN,Units
1,Cambodia,"Debt instruments, Central government, Percent ...",Annual,1976,NaN,Units
2,Cambodia,"Debt instruments, Central government, Percent ...",Annual,1977,NaN,Units
3,Cambodia,"Debt instruments, Central government, Percent ...",Annual,1978,NaN,Units
4,Cambodia,"Debt instruments, Central government, Percent ...",Annual,1979,NaN,Units


In [ ]:
# display the data type of "TIME_PERIOD"
print(debt_to_gdp["TIME_PERIOD"].dtype)

int64


In [ ]:
gdd = (
    debt_to_gdp[["COUNTRY", "TIME_PERIOD", "OBS_VALUE"]]
    .rename(columns={
        "TIME_PERIOD": "TIME",
        "OBS_VALUE":   "DEBT_GDP"
    })
    .assign(
        COUNTRY  = lambda x: x["COUNTRY"].str.strip(),
        TIME     = lambda x: pd.to_numeric(x["TIME"],     errors="coerce").astype("Int64"),
        DEBT_GDP = lambda x: pd.to_numeric(x["DEBT_GDP"], errors="coerce"),
    )
    .dropna(subset=["COUNTRY", "TIME"])
    .drop_duplicates(subset=["COUNTRY", "TIME"], keep="first")
    .sort_values(["COUNTRY", "TIME"])
    .reset_index(drop=True)
)

In [ ]:
display(gdd)

,COUNTRY,TIME,DEBT_GDP
0,"Afghanistan, Islamic Republic of",1975,NaN
1,"Afghanistan, Islamic Republic of",1976,NaN
2,"Afghanistan, Islamic Republic of",1977,NaN
3,"Afghanistan, Islamic Republic of",1978,NaN
4,"Afghanistan, Islamic Republic of",1979,NaN
...,...,...,...
8735,Zimbabwe,2016,49.91
8736,Zimbabwe,2017,68.93
8737,Zimbabwe,2018,48.12
8738,Zimbabwe,2019,82.34


In [ ]:
rows = []
for _, row in gdd.iterrows():
    for month in range(1, 13):
        rows.append({
            "COUNTRY":  row["COUNTRY"],
            "TIME":     f"{int(row['TIME'])}-M{month:02d}",
            "DEBT_GDP": row["DEBT_GDP"],
        })

gdd_monthly = (
    pd.DataFrame(rows)
    .sort_values(["COUNTRY", "TIME"])
    .reset_index(drop=True)
)

In [ ]:
# export gdd_monthly to excel
gdd_monthly.to_excel('gdd_monthly.xlsx', index=False)

# Import us rates

In [ ]:
# us_rates
display(us_rates.head())

,observation_date,GS10
0,1975-01-01,7.50
1,1975-02-01,7.39
2,1975-03-01,7.73
3,1975-04-01,8.23
4,1975-05-01,8.06


In [ ]:
# Convert 'observation_date' to datetime objects
us_rates['observation_date'] = pd.to_datetime(us_rates['observation_date'])

# Create the new 'TIME' column in 'yyyy-Mmm' format
us_rates['TIME'] = us_rates['observation_date'].dt.strftime('%Y-M%m')

# Display the head of the DataFrame with the new 'TIME' column
display(us_rates.head())

,observation_date,GS10,TIME
0,1975-01-01,7.50,1975-M01
1,1975-02-01,7.39,1975-M02
2,1975-03-01,7.73,1975-M03
3,1975-04-01,8.23,1975-M04
4,1975-05-01,8.06,1975-M05


# Modify the trade dataset
# create import and export variable

In [ ]:
# trade
display(trade.head())

,COUNTRY,INDICATOR,COUNTERPART_COUNTRY,FREQUENCY,TIME_PERIOD,OBS_VALUE,SCALE
0,Romania,"Imports of goods, Free on board (FOB), US dollar",World,Monthly,1981-M01,936.013391,Millions
1,Romania,"Imports of goods, Free on board (FOB), US dollar",World,Monthly,1981-M02,974.216451,Millions
2,Romania,"Imports of goods, Free on board (FOB), US dollar",World,Monthly,1981-M03,936.464667,Millions
3,Romania,"Imports of goods, Free on board (FOB), US dollar",World,Monthly,1981-M04,1064.354191,Millions
4,Romania,"Imports of goods, Free on board (FOB), US dollar",World,Monthly,1981-M05,969.188267,Millions


In [ ]:
countries_to_drop = [
    'Advanced Economies',
    'Africa'
    'CIS',
    'EMDEs by Source of Export Earnings: Fuel',
    'EMDEs by Source of Export Earnings: Nonfuel',
    'Emerging and Developing Asia',
    'Emerging and Developing Europe',
    'Emerging Market and Developing Economies',
    'Euro Area (EA)',
    'Europe',
    'European Union (EU)',
    'Holy See',
    'Middle East',
    'Middle East and Central Asia',
    'Middle East, North Africa, Afghanistan, and Pakistan',
    'Sub-Saharan Africa (SSA)',
    'World'
]

trade = trade[~trade['COUNTRY'].isin(countries_to_drop)]

In [ ]:
df = trade

# Imports filtern
imports = (
    df[df["INDICATOR"] == "Imports of goods, Free on board (FOB), US dollar"]
    [["COUNTRY", "TIME_PERIOD", "OBS_VALUE"]]
    .rename(columns={"OBS_VALUE": "IMPORTS from WORLD"})
)

# Exports filtern
exports = (
    df[df["INDICATOR"] == "Exports of goods, Free on board (FOB), US dollar"]
    [["COUNTRY", "TIME_PERIOD", "OBS_VALUE"]]
    .rename(columns={"OBS_VALUE": "EXPORTS to WORLD"})
)

# Zusammenführen
result = pd.merge(imports, exports, on=["COUNTRY", "TIME_PERIOD"], how="outer")

# Spalte umbenennen (optional, falls du TIME statt TIME_PERIOD willst)
result = result.rename(columns={"TIME_PERIOD": "TIME"})

In [ ]:
display(result)

,COUNTRY,TIME,IMPORTS from WORLD,EXPORTS to WORLD
0,"Afghanistan, Islamic Republic of",1981-M01,NaN,66.203470
1,"Afghanistan, Islamic Republic of",1981-M02,NaN,64.097939
2,"Afghanistan, Islamic Republic of",1981-M03,NaN,61.466963
3,"Afghanistan, Islamic Republic of",1981-M04,NaN,60.295731
4,"Afghanistan, Islamic Republic of",1981-M05,NaN,37.498092
...,...,...,...,...
95582,Zimbabwe,2020-M08,557.525540,389.317141
95583,Zimbabwe,2020-M09,635.198937,398.855116
95584,Zimbabwe,2020-M10,690.875271,439.431767
95585,Zimbabwe,2020-M11,693.830735,528.354453


In [ ]:
trade = result

In [ ]:
trade = trade.rename(columns={'TIME_PERIOD': 'TIME'})

# Modify exchange rates dataset

In [ ]:
display(exchange_rates.head())

,COUNTRY,INDICATOR,TYPE_OF_TRANSFORMATION,FREQUENCY,TIME_PERIOD,OBS_VALUE,SCALE
0,Myanmar,US Dollar per domestic currency,End-of-period (EoP),Monthly,1975-M01,0.158612,Units
1,Myanmar,US Dollar per domestic currency,End-of-period (EoP),Monthly,1975-M02,0.161288,Units
2,Myanmar,US Dollar per domestic currency,End-of-period (EoP),Monthly,1975-M03,0.159612,Units
3,Myanmar,US Dollar per domestic currency,End-of-period (EoP),Monthly,1975-M04,0.160583,Units
4,Myanmar,US Dollar per domestic currency,End-of-period (EoP),Monthly,1975-M05,0.157436,Units


In [ ]:
# rename columns TIME_PERIOD to TIME and OBS_VALUE to er_rate
exchange_rates = exchange_rates.rename(columns={'TIME_PERIOD': 'TIME', 'OBS_VALUE': 'er_rate'})

In [ ]:
# drop INDICATOR, TYPE_OF_TRANSFORMATION, FREQUENCY and SCALE
exchange_rates = exchange_rates.drop(columns=['INDICATOR', 'TYPE_OF_TRANSFORMATION', 'FREQUENCY', 'SCALE'])

# Modify CPI dataset

In [ ]:
display(cpi_data.head())

,COUNTRY,INDEX_TYPE,COICOP_1999,TYPE_OF_TRANSFORMATION,FREQUENCY,TIME_PERIOD,OBS_VALUE,SCALE
0,United Kingdom,Consumer price index (CPI),All Items,"Standard reference period (2010=100), Index",Monthly,1975-M01,15.311585,Units
1,United Kingdom,Consumer price index (CPI),All Items,"Standard reference period (2010=100), Index",Monthly,1975-M02,15.566998,Units
2,United Kingdom,Consumer price index (CPI),All Items,"Standard reference period (2010=100), Index",Monthly,1975-M03,15.873479,Units
3,United Kingdom,Consumer price index (CPI),All Items,"Standard reference period (2010=100), Index",Monthly,1975-M04,16.486462,Units
4,United Kingdom,Consumer price index (CPI),All Items,"Standard reference period (2010=100), Index",Monthly,1975-M05,17.176057,Units


In [ ]:
# rename columns TIME_PERIOD to TIME and OBS_VALUE to er_rate
cpi_data = cpi_data.rename(columns={'TIME_PERIOD': 'TIME', 'OBS_VALUE': 'cpi'})

In [ ]:
# drop INDEX_TYPE, COICOP_1999, TYPE_OF_TRANSFORMATION, FREQUENCY and SCALE
cpi_data = cpi_data.drop(columns=['INDEX_TYPE', 'COICOP_1999', 'TYPE_OF_TRANSFORMATION', 'FREQUENCY', 'SCALE'])

In [ ]:
display(cpi_data.head())

,COUNTRY,TIME,cpi
0,United Kingdom,1975-M01,15.311585
1,United Kingdom,1975-M02,15.566998
2,United Kingdom,1975-M03,15.873479
3,United Kingdom,1975-M04,16.486462
4,United Kingdom,1975-M05,17.176057


# Standardize Country Names in `sov_default`

In [ ]:
# default data
display(sov_default.head())

,Unnamed: 0,Case nr,New case? (compared to 2016 JEEA paper),Case nr in Cruces/Trebesch database (2014 update),COUNTRY,WDI code,Start of default or restructuring process: default or announcement,End of restructuring: completion of exchange,Alternative end date / follow-up restructurings,Strictly preemptive,Weakly preemptive,Post-default,Default date,Announcement,No exact start date (missing default month),TIME
0,NaN,1,NaN,1.0,Albania,ALB,1991-11-01,1995-08-31,NaN,0,0,1,1991-11-01,1991-11-01,0,1991-M11
1,NaN,2,NaN,2.0,Algeria,DZA,1990-10-01,1992-03-01,NaN,1,1,0,NaT,1990-10-01,0,1990-M10
2,NaN,3,NaN,3.0,Algeria,DZA,1993-12-01,1996-07-17,NaN,0,0,1,1994-03-01,1993-12-01,0,1993-M12
3,NaN,4,NaN,4.0,Argentina,ARG,1982-07-01,1985-08-27,NaN,0,0,1,1983-11-01,1983-11-01,0,1982-M07
4,NaN,5,NaN,5.0,Argentina,ARG,1985-08-01,1987-08-21,NaN,0,1,0,1985-08-01,1985-08-01,0,1985-M08


In [ ]:
sov_default.rename(columns={'Country case': 'COUNTRY'}, inplace=True)


In [ ]:
country_replacements = {
    'Chad (Glencore loans)': 'Chad',
    'Croatia': 'Croatia, Republic of',
    'Madagascar': 'Madagascar, Republic of',
    'Mauritania': 'Mauritania, Islamic Republic of',
    'Moldova (Eurobonds)': 'Moldova, Republic of',
    'Mozambique': 'Mozambique, Republic of',
    'Pakistan (bank debt)': 'Pakistan',
    'Panama': 'Panama',
    'Poland': 'Poland, Republic of',
    'Russia': 'Russian Federation',
    'Sao Tome and Principe': 'São Tomé and Príncipe, Democratic Republic of',
    'Serbia and Montenegro': 'Serbia, Republic of',
    'Slovenia': 'Slovenia, Republic of',
    'Tanzania': 'Tanzania, United Republic of',
    'Turkey': 'Türkiye, Republic of',
    'Ukraine (Chase loan)': 'Ukraine',
    'Venezuela, RB': 'Venezuela, República Bolivariana de'
}

sov_default['COUNTRY'] = sov_default['COUNTRY'].replace(country_replacements)
display(sov_default.head())

,Unnamed: 0,Case nr,New case? (compared to 2016 JEEA paper),Case nr in Cruces/Trebesch database (2014 update),COUNTRY,WDI code,Start of default or restructuring process: default or announcement,End of restructuring: completion of exchange,Alternative end date / follow-up restructurings,Strictly preemptive,Weakly preemptive,Post-default,Default date,Announcement,No exact start date (missing default month)
0,NaN,1,NaN,1.0,Albania,ALB,1991-11-01,1995-08-31,NaN,0,0,1,1991-11-01,1991-11-01,0
1,NaN,2,NaN,2.0,Algeria,DZA,1990-10-01,1992-03-01,NaN,1,1,0,NaT,1990-10-01,0
2,NaN,3,NaN,3.0,Algeria,DZA,1993-12-01,1996-07-17,NaN,0,0,1,1994-03-01,1993-12-01,0
3,NaN,4,NaN,4.0,Argentina,ARG,1982-07-01,1985-08-27,NaN,0,0,1,1983-11-01,1983-11-01,0
4,NaN,5,NaN,5.0,Argentina,ARG,1985-08-01,1987-08-21,NaN,0,1,0,1985-08-01,1985-08-01,0


### Create 'TIME' column for `sov_default`

In [ ]:
# Create the new 'TIME' column in 'yyyy-Mmm' format
sov_default['TIME'] = sov_default['Start of default or restructuring process: default or announcement'].dt.strftime('%Y-M%m')

# Display the head of the DataFrame with the new 'TIME' column
display(sov_default.head())

,Unnamed: 0,Case nr,New case? (compared to 2016 JEEA paper),Case nr in Cruces/Trebesch database (2014 update),COUNTRY,WDI code,Start of default or restructuring process: default or announcement,End of restructuring: completion of exchange,Alternative end date / follow-up restructurings,Strictly preemptive,Weakly preemptive,Post-default,Default date,Announcement,No exact start date (missing default month),TIME
0,NaN,1,NaN,1.0,Albania,ALB,1991-11-01,1995-08-31,NaN,0,0,1,1991-11-01,1991-11-01,0,1991-M11
1,NaN,2,NaN,2.0,Algeria,DZA,1990-10-01,1992-03-01,NaN,1,1,0,NaT,1990-10-01,0,1990-M10
2,NaN,3,NaN,3.0,Algeria,DZA,1993-12-01,1996-07-17,NaN,0,0,1,1994-03-01,1993-12-01,0,1993-M12
3,NaN,4,NaN,4.0,Argentina,ARG,1982-07-01,1985-08-27,NaN,0,0,1,1983-11-01,1983-11-01,0,1982-M07
4,NaN,5,NaN,5.0,Argentina,ARG,1985-08-01,1987-08-21,NaN,0,1,0,1985-08-01,1985-08-01,0,1985-M08


### Create extended `sov_default_ext` DataFrame with full time series

In [ ]:
# 1. Get unique countries from sov_default
unique_countries = sov_default['COUNTRY'].unique()

# 2. Generate a complete monthly time range from 1975-01 to 2020-12
start_date = '1975-01-01'
end_date = '2020-12-01'
full_time_range = pd.date_range(start=start_date, end=end_date, freq='MS')

# Format the time range to 'YYYY-Mmm'
full_time_range_formatted = full_time_range.strftime('%Y-M%m').tolist()

# 3. Create a Cartesian product of unique countries and the full time range
from itertools import product
all_combinations = pd.DataFrame(list(product(unique_countries, full_time_range_formatted)), columns=['COUNTRY', 'TIME'])

# Create a 'TIME_DT' column for easier comparisons in the next step
all_combinations['TIME_DT'] = pd.to_datetime(all_combinations['TIME'], format='%Y-M%m')

# Initialize the columns that will hold the 'Start of default' and 'End of restructuring' dates for each month
# These will be filled based on active default spells
all_combinations['Start of default or restructuring process: default or announcement'] = pd.NaT
all_combinations['End of restructuring: completion of exchange'] = pd.NaT

# Prepare the original sov_default to have datetime objects for event dates
sov_default_events = sov_default.copy()
sov_default_events['Start_DT_Event'] = pd.to_datetime(sov_default_events['Start of default or restructuring process: default or announcement'], errors='coerce')
sov_default_events['End_DT_Event'] = pd.to_datetime(sov_default_events['End of restructuring: completion of exchange'], errors='coerce')

# Iterate over each unique country
for country in unique_countries:
    # Get all default events for the current country
    country_defaults = sov_default_events[sov_default_events['COUNTRY'] == country]

    # Get the mask for the current country in all_combinations
    country_mask_all_combinations = (all_combinations['COUNTRY'] == country)

    # For each default event of this country
    for _, event_row in country_defaults.iterrows():
        start_event_dt = event_row['Start_DT_Event']
        end_event_dt = event_row['End_DT_Event']

        if pd.isna(start_event_dt):
            continue # Skip invalid start dates for events

        # Determine the mask for time periods within this specific default event
        is_within_event = (all_combinations.loc[country_mask_all_combinations, 'TIME_DT'] >= start_event_dt) & \
                          (
                              (pd.isna(end_event_dt)) | # If end_event_dt is NaT, default is considered ongoing until a valid end_date is encountered
                              (all_combinations.loc[country_mask_all_combinations, 'TIME_DT'] <= end_event_dt)
                          )

        # Update the 'Start of default...' and 'End of restructuring...' columns for these months
        all_combinations.loc[country_mask_all_combinations & is_within_event, 'Start of default or restructuring process: default or announcement'] = start_event_dt
        all_combinations.loc[country_mask_all_combinations & is_within_event, 'End of restructuring: completion of exchange'] = end_event_dt

flag_cols = ['COUNTRY',
             'Start of default or restructuring process: default or announcement',
             'Strictly preemptive ', 'Weakly preemptive ', 'Post-default']

flags = sov_default_events[flag_cols].copy()

sov_default_ext = all_combinations.merge(
    flags,
    on=['COUNTRY', 'Start of default or restructuring process: default or announcement'],
    how='left'
)

# Drop the temporary 'TIME_DT' column
sov_default_ext = sov_default_ext.drop(columns=['TIME_DT'])

# Fill NaN flags with 0 for non-default months
sov_default_ext[['Strictly preemptive ', 'Weakly preemptive ', 'Post-default']] = \
    sov_default_ext[['Strictly preemptive ', 'Weakly preemptive ', 'Post-default']].fillna(0)

display(sov_default_ext.head())
print(f"Shape of sov_default_ext: {sov_default_ext.shape}")

,COUNTRY,TIME,Start of default or restructuring process: default or announcement,End of restructuring: completion of exchange,Strictly preemptive,Weakly preemptive,Post-default
0,Albania,1975-M01,NaT,NaT,0.0,0.0,0.0
1,Albania,1975-M02,NaT,NaT,0.0,0.0,0.0
2,Albania,1975-M03,NaT,NaT,0.0,0.0,0.0
3,Albania,1975-M04,NaT,NaT,0.0,0.0,0.0
4,Albania,1975-M05,NaT,NaT,0.0,0.0,0.0


Shape of sov_default_ext: (50901, 7)


In [ ]:
# Ensure 'TIME' column in sov_default_ext is a datetime object for comparison
sov_default_ext['TIME_DT'] = pd.to_datetime(sov_default_ext['TIME'], format='%Y-M%m', errors='coerce')

# Ensure 'Start of default or restructuring process: default or announcement' and 'End of restructuring: completion of exchange' are datetime objects
sov_default_ext['Start of default or restructuring process: default or announcement_DT'] = pd.to_datetime(sov_default_ext['Start of default or restructuring process: default or announcement'], errors='coerce')
sov_default_ext['End of restructuring: completion of exchange_DT'] = pd.to_datetime(sov_default_ext['End of restructuring: completion of exchange'], errors='coerce')

# Create the 'DEFAULT' dummy variable
sov_default_ext['DEFAULT'] = (
    (sov_default_ext['TIME_DT'] >= sov_default_ext['Start of default or restructuring process: default or announcement_DT']) &
    (
        sov_default_ext['End of restructuring: completion of exchange_DT'].isnull() |
        (sov_default_ext['TIME_DT'] <= sov_default_ext['End of restructuring: completion of exchange_DT'])
    )
).astype(int)

# Display the head of sov_default_ext with the new 'DEFAULT' column
display(sov_default_ext[['COUNTRY', 'TIME', 'Start of default or restructuring process: default or announcement', 'End of restructuring: completion of exchange', 'DEFAULT', 'Strictly preemptive ', 'Weakly preemptive ', 'Post-default']])

,COUNTRY,TIME,Start of default or restructuring process: default or announcement,End of restructuring: completion of exchange,DEFAULT,Strictly preemptive,Weakly preemptive,Post-default
0,Albania,1975-M01,NaT,NaT,0,0.0,0.0,0.0
1,Albania,1975-M02,NaT,NaT,0,0.0,0.0,0.0
2,Albania,1975-M03,NaT,NaT,0,0.0,0.0,0.0
3,Albania,1975-M04,NaT,NaT,0,0.0,0.0,0.0
4,Albania,1975-M05,NaT,NaT,0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
50896,Zambia,2020-M08,NaT,NaT,0,0.0,0.0,0.0
50897,Zambia,2020-M09,NaT,NaT,0,0.0,0.0,0.0
50898,Zambia,2020-M10,NaT,NaT,0,0.0,0.0,0.0
50899,Zambia,2020-M11,NaT,NaT,0,0.0,0.0,0.0


In [ ]:
display(sov_default_ext.head())

,COUNTRY,TIME,Start of default or restructuring process: default or announcement,End of restructuring: completion of exchange,TIME_DT,Start of default or restructuring process: default or announcement_DT,End of restructuring: completion of exchange_DT,DEFAULT
0,Albania,1975-M01,NaT,NaT,1975-01-01,NaT,NaT,0
1,Albania,1975-M02,NaT,NaT,1975-02-01,NaT,NaT,0
2,Albania,1975-M03,NaT,NaT,1975-03-01,NaT,NaT,0
3,Albania,1975-M04,NaT,NaT,1975-04-01,NaT,NaT,0
4,Albania,1975-M05,NaT,NaT,1975-05-01,NaT,NaT,0


# Merge Datasets

In [ ]:
# overview of trade, us_rates, gdd_monthly and sov_default_ext
print(f"Shape of trade: {trade.shape}")
print(f"Shape of us_rates: {us_rates.shape}")
print(f"Shape of gdd_monthly: {gdd_monthly.shape}")
print(f"Shape of sov_default_ext: {sov_default_ext.shape}")
print(f"Shape of exchange_rates: {exchange_rates.shape}")
print(f"Shape of cpi_data: {cpi_data.shape}")

Shape of trade: (95587, 4)
Shape of us_rates: (553, 3)
Shape of gdd_monthly: (104880, 3)
Shape of sov_default_ext: (50901, 11)
Shape of exchange_rates: (95723, 3)
Shape of cpi_data: (78820, 3)


In [ ]:
# =============================================================================
# Schritt 1: Gemeinsamen Länder-Zeit-Rahmen aus allen drei Quellen aufbauen
# =============================================================================
frame_default = sov_default_ext[['COUNTRY', 'TIME']].dropna(subset=['COUNTRY'])
frame_trade   = trade[['COUNTRY', 'TIME']].dropna(subset=['COUNTRY'])
frame_gdd     = gdd_monthly[['COUNTRY', 'TIME']].dropna(subset=['COUNTRY'])

full_frame = (
    pd.concat([frame_default, frame_trade, frame_gdd], ignore_index=True)
    .drop_duplicates()
    .sort_values(['COUNTRY', 'TIME'])
    .reset_index(drop=True)
)

print(f"Shape of full_frame (Länder-Zeit-Anker): {full_frame.shape}")


Shape of full_frame (Länder-Zeit-Anker): (127452, 2)


In [ ]:
# =============================================================================
# Schritt 2: Default-Info drauf mergen
# =============================================================================
merged = pd.merge(full_frame, sov_default_ext, on=['COUNTRY', 'TIME'], how='left')
merged['DEFAULT'] = merged['DEFAULT'].fillna(0)
print(f"Shape after default merge: {merged.shape}")

Shape after default merge: (127569, 11)


In [ ]:
# =============================================================================
# Schritt 3: GS10 drauf mergen (nur TIME – gilt für alle Länder gleich)
# =============================================================================
merged = pd.merge(merged, us_rates[['TIME', 'GS10']], on='TIME', how='left')
print(f"Shape after GS10 merge: {merged.shape}")

Shape after GS10 merge: (127569, 12)


In [ ]:
# =============================================================================
# Schritt 4: Trade drauf mergen
# =============================================================================
merged = pd.merge(merged, trade, on=['COUNTRY', 'TIME'], how='left', suffixes=('', '_trade'))
print(f"Shape after trade merge: {merged.shape}")


Shape after trade merge: (127569, 14)


In [ ]:
# =============================================================================
# Schritt 5: GDD drauf mergen
# =============================================================================
merged = pd.merge(merged, gdd_monthly, on=['COUNTRY', 'TIME'], how='left', suffixes=('', '_gdd'))
print(f"Shape after GDD merge: {merged.shape}")


Shape after GDD merge: (127569, 15)


In [ ]:
# =============================================================================
# Schritt 6: exchange_rates drauf mergen
# =============================================================================
merged = pd.merge(merged, exchange_rates, on=['COUNTRY', 'TIME'], how='left', suffixes=('', '_er_rates'))
print(f"Shape after exchange_rates merge: {merged.shape}")


Shape after exchange_rates merge: (127569, 16)


In [ ]:
# =============================================================================
# Schritt 7: cpi_data drauf mergen
# =============================================================================
merged_default_trade_rates_gdd_er_cpi = pd.merge(merged, cpi_data, on=['COUNTRY', 'TIME'], how='left', suffixes=('', '_cpi'))
print(f"Shape after cpi_data merge: {merged.shape}")

Shape after cpi_data merge: (127569, 16)


In [ ]:
# fill out empty values for countries not included in default data (Trebesch et al.)
merged_default_trade_rates_gdd_er_cpi['DEFAULT'].fillna(0, inplace=True)

/tmp/ipykernel_1615/987204369.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_default_trade_rates_gdd_er_cpi['DEFAULT'].fillna(0, inplace=True)


In [ ]:
# check for duplicates
num_duplicates = merged_default_trade_rates_gdd_er_cpi.duplicated().sum()
print(f"Number of duplicate rows in merged_default_trade_rates_gdd_er_cpi: {num_duplicates}")

# Optionally, display the duplicate rows if there are any
if num_duplicates > 0:
    display(merged_default_trade_rates_gdd_er_cpi[merged_default_trade_rates_gdd_er_cpi.duplicated(keep=False)])


Number of duplicate rows in merged_default_trade_rates_gdd_er_cpi: 117


,COUNTRY,TIME,Start of default or restructuring process: default or announcement,End of restructuring: completion of exchange,Strictly preemptive,Weakly preemptive,Post-default,TIME_DT,Start of default or restructuring process: default or announcement_DT,End of restructuring: completion of exchange_DT,DEFAULT,GS10,IMPORTS from WORLD,EXPORTS to WORLD,DEBT_GDP,er_rate,cpi
15269,Brazil,1989-M06,1989-06-01,1994-04-15,0.0,0.0,1.0,1989-06-01,1989-06-01,1994-04-15,1.0,8.28,1624.89,3189.915934,33.80,1.810402e+06,0.000014
15270,Brazil,1989-M06,1989-06-01,1994-04-15,0.0,0.0,1.0,1989-06-01,1989-06-01,1994-04-15,1.0,8.28,1624.89,3189.915934,33.80,1.810402e+06,0.000014
15271,Brazil,1989-M07,1989-06-01,1994-04-15,0.0,0.0,1.0,1989-07-01,1989-06-01,1994-04-15,1.0,8.02,1917.39,2966.672321,33.80,1.269621e+06,0.000018
15272,Brazil,1989-M07,1989-06-01,1994-04-15,0.0,0.0,1.0,1989-07-01,1989-06-01,1994-04-15,1.0,8.02,1917.39,2966.672321,33.80,1.269621e+06,0.000018
15273,Brazil,1989-M08,1989-06-01,1994-04-15,0.0,0.0,1.0,1989-08-01,1989-06-01,1994-04-15,1.0,8.11,2062.29,3071.639199,33.80,9.814418e+05,0.000024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116059,"Türkiye, Republic of",1982-M01,1981-01-01,1982-03-13,0.0,1.0,0.0,1982-01-01,1981-01-01,1982-03-13,1.0,14.59,NaN,446.130000,29.45,7.077191e+03,0.002793
116060,"Türkiye, Republic of",1982-M02,1981-01-01,1982-03-13,0.0,1.0,0.0,1982-02-01,1981-01-01,1982-03-13,1.0,14.43,NaN,410.120000,29.45,6.906841e+03,0.002856
116061,"Türkiye, Republic of",1982-M02,1981-01-01,1982-03-13,0.0,1.0,0.0,1982-02-01,1981-01-01,1982-03-13,1.0,14.43,NaN,410.120000,29.45,6.906841e+03,0.002856
116062,"Türkiye, Republic of",1982-M03,1981-01-01,1982-03-13,0.0,1.0,0.0,1982-03-01,1981-01-01,1982-03-13,1.0,13.86,NaN,447.270000,29.45,6.756072e+03,0.002922


In [ ]:
# drop duplicates of merged_default_trade_rates_gdd_er_cpi
merged_default_trade_rates_gdd_er_cpi = merged_default_trade_rates_gdd_er_cpi.drop_duplicates()

In [ ]:
empty_country_rows = merged_default_trade_rates_gdd_er_cpi[merged_default_trade_rates_gdd_er_cpi['COUNTRY'].isnull() | (merged_default_trade_rates_gdd_er_cpi['COUNTRY'] == '')]
print(f"Number of rows with empty 'COUNTRY': {len(empty_country_rows)}")

if not empty_country_rows.empty:
    display(empty_country_rows.head())


Number of rows with empty 'COUNTRY': 0


In [ ]:
# Fill NaN flags with 0 for non-default months for extended dataset
merged_default_trade_rates_gdd_er_cpi[['Strictly preemptive ', 'Weakly preemptive ', 'Post-default']] = \
    merged_default_trade_rates_gdd_er_cpi[['Strictly preemptive ', 'Weakly preemptive ', 'Post-default']].fillna(0)

In [ ]:
analysis_df = merged_default_trade_rates_gdd_er_cpi[['COUNTRY', 'TIME',
                                          'Start of default or restructuring process: default or announcement',
                                          'End of restructuring: completion of exchange', 'DEFAULT',
                                          'Strictly preemptive ', 'Weakly preemptive ', 'Post-default',
                                          'IMPORTS from WORLD', 'EXPORTS to WORLD', 'GS10', 'DEBT_GDP',
                                          'er_rate', 'cpi']]

In [ ]:
# Drop observations if COUNTRY = 'Africa'
analysis_df = analysis_df[analysis_df['COUNTRY'] != 'Africa']

# display how many observations where dropped
print(f"Number of observations dropped: {len(merged_default_trade_rates_gdd_er_cpi) - len(analysis_df)}")

Number of observations dropped: 552


In [ ]:
#analysis_df.to_excel('analysis_df_check.xlsx', index=False)
#print("DataFrame exported to 'analysis_df.xlsx'")

# Fix country merge
The sovereign default dataset gives als information about the type of defaulting debt for some observations. Due to that the country names are different and won't merge. For this reason, for all observations with default type information the values of the country-time-specific are applied.

In [ ]:
# =============================================================================
# Mapping: debt-type country name  -->  standard country name
# =============================================================================
COUNTRY_MAPPING = {
    "Dominican Rep. (bank debt)":               "Dominican Rep.",
    "Dominican Rep. (bond debt)":               "Dominican Rep.",
    "Moldova (Gazprom debt)":                   "Moldova",
    "Republic of Mozambique (EMATUM Notes)":    "Mozambique, Republic of",
    "Republic of Mozambique (Eurobonds)":       "Mozambique, Republic of",
    "Pakistan (bond debt)":                     "Pakistan",
    "Panama (bond debt)":                       "Panama",
    "Russia (GKOs, non-resid.)":               "Russian Federation",
    "Russia (MinFin3)":                         "Russian Federation",
    "Russia (PRINs & IANs)":                   "Russian Federation",
    "Ukraine (Commercial loans)":               "Ukraine",
    "Ukraine (Eurobonds)":                      "Ukraine",
    "Ukraine (Global Exchange)":                "Ukraine",
    "Ukraine (ING debt / Merrill Lynch)":       "Ukraine",
    "Ukraine (OVDPs, non-resid.)":             "Ukraine",
}

VALUE_COLS = ["IMPORTS from WORLD", "EXPORTS to WORLD", "GS10", "DEBT_GDP", "er_rate", "cpi"]


In [ ]:
# =============================================================================
# Step 1: Build a lookup table from the STANDARD country rows
#         (i.e. rows whose COUNTRY is NOT a debt-type variant)
# =============================================================================
df = analysis_df.copy() # Ensure df refers to the correct analysis_df

standard_rows = df[~df["COUNTRY"].isin(COUNTRY_MAPPING.keys())].copy()

lookup = (
    standard_rows
    .groupby(["COUNTRY", "TIME"])[VALUE_COLS]
    .first()          # one value per country-time from the standard rows
    .reset_index()
)

In [ ]:
# =============================================================================
# Step 2: For each debt-type row, add a temporary column with the standard
#         country name so we can join against the lookup table
# =============================================================================
df["_standard_country"] = df["COUNTRY"].map(COUNTRY_MAPPING)   # NaN for non-debt rows

# Merge lookup values onto ALL rows (joining on standard name + TIME)
df = df.merge(
    lookup.rename(columns={col: f"_ref_{col}" for col in VALUE_COLS}),
    left_on=["_standard_country", "TIME"],
    right_on=["COUNTRY", "TIME"],
    how="left",
    suffixes=("", "_lookup"),
)

# Drop the extra COUNTRY column that came from the lookup rename
if "COUNTRY_lookup" in df.columns:
    df.drop(columns=["COUNTRY_lookup"], inplace=True)

In [ ]:
# =============================================================================
# Step 3: Fill missing values in VALUE_COLS using the reference values –
#         only for debt-type rows (standard rows are untouched)
# =============================================================================
for col in VALUE_COLS:
    ref_col = f"_ref_{col}"
    # Only fill where COUNTRY is a debt-type variant
    debt_mask = df["COUNTRY"].isin(COUNTRY_MAPPING.keys())
    df.loc[debt_mask, col] = df.loc[debt_mask, col].fillna(df.loc[debt_mask, ref_col])
    df.drop(columns=[ref_col], inplace=True)

# Drop helper column
df.drop(columns=["_standard_country"], inplace=True)


In [ ]:
# Sanity check: show a few debt-type rows after filling
print("\nSample debt-type rows after filling:")
print(
    df[df["COUNTRY"].isin(COUNTRY_MAPPING.keys())]
    [["COUNTRY", "TIME"] + VALUE_COLS]
    .head(20)
    .to_string(index=False)
)


Sample debt-type rows after filling:
                   COUNTRY     TIME  IMPORTS from WORLD  EXPORTS to WORLD  GS10  DEBT_GDP  er_rate  cpi
Dominican Rep. (bank debt) 1975-M01                 NaN               NaN  7.50       NaN      NaN  NaN
Dominican Rep. (bank debt) 1975-M02                 NaN               NaN  7.39       NaN      NaN  NaN
Dominican Rep. (bank debt) 1975-M03                 NaN               NaN  7.73       NaN      NaN  NaN
Dominican Rep. (bank debt) 1975-M04                 NaN               NaN  8.23       NaN      NaN  NaN
Dominican Rep. (bank debt) 1975-M05                 NaN               NaN  8.06       NaN      NaN  NaN
Dominican Rep. (bank debt) 1975-M06                 NaN               NaN  7.86       NaN      NaN  NaN
Dominican Rep. (bank debt) 1975-M07                 NaN               NaN  8.06       NaN      NaN  NaN
Dominican Rep. (bank debt) 1975-M08                 NaN               NaN  8.40       NaN      NaN  NaN
Dominican Rep. (bank debt)

In [ ]:
analysis_df = df.copy()

In [ ]:
analysis_df.to_excel('analysis_df.xlsx', index=False)
print("DataFrame exported to 'analysis_df.xlsx'")

DataFrame exported to 'analysis_df.xlsx'
